# Session VII: Command Line Tools and Automation

Use Python to automate tasks and interact through the command line, including parsing arguments and running scripts from the terminal.

## Exercise 1: CLI Greeter and Calculator
**Objective:** Practice basic command-line parsing using a list that represents `sys.argv`.

Instead of reading real terminal input in the notebook, you will write a function that receives an argument list like:

```python
["greet", "Alice"]
["add", "3", "4"]
```

**Tasks:**
1. `run_command(argv)` should support two commands:
   - `greet <name>` → return `"Hello, <name>!"`
   - `add <x> <y>` → return the numeric sum as a string
2. Return a helpful usage message for missing or invalid input.
3. Use `try/except` to handle non-numeric values in `add`.


In [1]:
def run_command(argv):
    """Parse a simple command list and return a response string."""
    usage = "Usage: greet <name> | add <x> <y>"

    # Handle empty input
    if not argv:
        return f"Error: no command given. {usage}"

    command = argv[0]

    # Support: greet <name>
    if command == "greet":
        if len(argv) != 2:
            return "Usage: greet <name>"
        return f"Hello, {argv[1]}!"

    # Support: add <x> <y>
    if command == "add":
        if len(argv) != 3:
            return "Usage: add <x> <y>"
        # Use try/except for numeric conversion
        try:
            x = float(argv[1])
            y = float(argv[2])
        except ValueError:
            return "Error: both operands of 'add' must be numbers"
        return str(x + y)

    return f"Error: unknown command '{command}'. {usage}"

# --- TEST ---
print(run_command(["greet", "Alice"]))
# Expected: Hello, Alice!

print(run_command(["add", "3", "4"]))
# Expected: 7.0

print(run_command(["add", "three", "4"]))
# Expected: Error message about numeric input

Hello, Alice!
7.0
Error: both operands of 'add' must be numbers


---
## Exercise 2: Expense Lister with `argparse`
**Objective:** Build a real argument parser using optional flags.

You are given a list of expense records. Build a command-line style filter that can:
- show all expenses
- filter by category
- filter by a minimum amount

**Tasks:**
1. `build_parser()` → Create an `ArgumentParser` with:
   - `--category`
   - `--min` (float, default `0`)
2. `filter_expenses(expenses, category=None, min_amount=0)` → Return only matching expenses.
3. `format_expenses(expenses)` → Return a formatted multi-line string.


In [ ]:
import argparse

expenses = [
    {"date": "2026-10-01", "category": "Food", "amount": 18.50, "note": "Lunch"},
    {"date": "2026-10-02", "category": "Transport", "amount": 2.80, "note": "Metro"},
    {"date": "2026-10-02", "category": "Food", "amount": 34.20, "note": "Groceries"},
    {"date": "2026-10-03", "category": "Books", "amount": 45.00, "note": "Python book"},
]

def build_parser():
    """Return an argparse parser for expense filtering."""
    # Create ArgumentParser
    parser = argparse.ArgumentParser(description="List and filter expenses")
    # Optional --category argument
    parser.add_argument("--category", type=str, help="Only show this category")
    # Optional --min argument
    parser.add_argument("--min", type=float, default=0, help="Minimum amount")
    return parser

def filter_expenses(expenses, category=None, min_amount=0):
    """Return filtered expenses matching the given criteria."""
    result = []
    for expense in expenses:
        # Skip rows from a different category (case-insensitive)
        if category and expense["category"].lower() != category.lower():
            continue
        # Skip rows below the minimum amount
        if expense["amount"] < min_amount:
            continue
        result.append(expense)
    return result

def format_expenses(expenses):
    """Return a formatted text report for a list of expenses."""
    # Handle empty list
    if not expenses:
        return "No expenses found."
    # Build a list of output lines
    lines = []
    for expense in expenses:
        lines.append(
            f"{expense['date']}  {expense['category']:<10} "
            f"{expense['amount']:>8.2f}  {expense['note']}"
        )
    # Return one multi-line string
    return "\n".join(lines)


# --- TEST ---
parser = build_parser()
args = parser.parse_args(["--category", "Food", "--min", "20"])
filtered = filter_expenses(expenses, args.category, args.min)
print(format_expenses(filtered))

---
## Exercise 3: Budget Backup Script
**Objective:** Automate data backups using `pathlib`, timestamps, and file copying.

Your script should validate that the source file exists, create a `backups` folder if needed, and make a timestamped copy.

**Tasks:**
1. `make_backup_name(filename)` → Return a new filename like `budget_20261005_224700.json`
2. `backup_file(source_path, backup_dir="backups")` → Copy the source file into the backup folder.
3. Return the destination path as a string, or `None` if the source file is missing.

**First, run the setup cell below to create a sample file.**


In [ ]:
from pathlib import Path
import json

sample_budget = [
    {"category": "Food", "amount": 20.5},
    {"category": "Transport", "amount": 15.0},
]

with open("budget_data.json", "w") as file:
    json.dump(sample_budget, file, indent=2)

print("Created budget_data.json")

In [ ]:
from pathlib import Path
from datetime import datetime
import shutil

def make_backup_name(filename):
    """Return a timestamped backup filename."""
    path = Path(filename)
    # Add a timestamp like 20261005_224700
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    return f"{path.stem}_{timestamp}{path.suffix}"

def backup_file(source_path, backup_dir="backups"):
    """Copy a file into backup_dir with a timestamped name."""
    # Convert to Path objects
    source = Path(source_path)
    backup_folder = Path(backup_dir)

    # Check whether source exists
    if not source.is_file():
        print(f"Error: '{source}' does not exist.")
        return None

    # Create backup folder if needed
    backup_folder.mkdir(parents=True, exist_ok=True)

    # Copy file with shutil.copy2 (keeps metadata)
    destination = backup_folder / make_backup_name(source.name)
    shutil.copy2(source, destination)
    return str(destination)


# --- TEST ---
result = backup_file("budget_data.json")
print("Backup created:", result)

---
## Exercise 4: CSV Expense Summary CLI
**Objective:** Load a CSV file, validate rows, and print category totals.

**Tasks:**
1. `load_expenses_csv(filename)` → Use `csv.DictReader` to load rows.
2. Skip malformed rows where:
   - `category` is empty
   - `amount` is missing or non-numeric
3. `summarize_by_category(expenses)` → Return a dictionary of totals.
4. `print_summary(summary)` → Print a formatted report.

**Run the setup cell below first.**


In [ ]:
import csv

rows = [
    ["date", "category", "amount", "note"],
    ["2026-10-01", "Food", "18.50", "Lunch"],
    ["2026-10-02", "Transport", "2.80", "Metro"],
    ["2026-10-02", "", "12.00", "Broken row"],
    ["2026-10-03", "Books", "forty", "Invalid amount"],
    ["2026-10-04", "Food", "21.30", "Dinner"],
]

with open("expenses_cli.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerows(rows)

print("Created expenses_cli.csv")

In [ ]:
import csv

def load_expenses_csv(filename):
    """Load valid expense rows from a CSV file."""
    expenses = []
    try:
        with open(filename, newline="") as file:
            reader = csv.DictReader(file)
            for line_number, row in enumerate(reader, start=2):
                category = (row.get("category") or "").strip()
                amount_text = (row.get("amount") or "").strip()

                # Validate category
                if not category:
                    print(f"Skipping line {line_number}: missing category")
                    continue

                # Validate and convert amount
                try:
                    amount = float(amount_text)
                except ValueError:
                    print(f"Skipping line {line_number}: invalid amount '{amount_text}'")
                    continue

                row["category"] = category
                row["amount"] = amount
                expenses.append(row)
    except FileNotFoundError:
        print(f"Error: '{filename}' not found.")
    return expenses

def summarize_by_category(expenses):
    """Return {category: total_amount}."""
    totals = {}
    for expense in expenses:
        category = expense["category"]
        totals[category] = totals.get(category, 0) + expense["amount"]
    return totals

def print_summary(summary):
    """Print a formatted category summary."""
    # Handle empty summary
    if not summary:
        print("No valid expenses to summarize.")
        return
    # Print aligned output
    print(f"{'Category':<12}{'Total':>10}")
    print("-" * 22)
    for category, total in sorted(summary.items()):
        print(f"{category:<12}{total:>10.2f}")
    print("-" * 22)
    print(f"{'TOTAL':<12}{sum(summary.values()):>10.2f}")


# --- TEST ---
expenses = load_expenses_csv("expenses_cli.csv")
summary = summarize_by_category(expenses)
print_summary(summary)

---
## Exercise 5: Scheduling Helper
**Objective:** Generate schedule strings for cron and Windows Task Scheduler.

You are **not** actually scheduling tasks in this notebook. Instead, you will generate the correct command strings.

**Tasks:**
1. `build_cron_schedule(minute, hour, script_path)` → Return a cron line like:
   `30 18 * * * python /path/to/script.py`
2. `build_task_scheduler_command(time_str, script_path)` → Return a Windows-style command string.
3. Validate the time values and raise or handle `ValueError` for invalid input.


In [ ]:
def build_cron_schedule(minute, hour, script_path):
    """Return a cron schedule line for a daily script."""
    # Validate minute (0-59) and hour (0-23)
    if not isinstance(minute, int) or not 0 <= minute <= 59:
        raise ValueError(f"Invalid minute: {minute} (must be 0-59)")
    if not isinstance(hour, int) or not 0 <= hour <= 23:
        raise ValueError(f"Invalid hour: {hour} (must be 0-23)")
    # Format: minute hour day-of-month month day-of-week command
    return f"{minute} {hour} * * * python {script_path}"

def build_task_scheduler_command(time_str, script_path):
    """Return a Windows schtasks command string."""
    # Validate HH:MM format
    try:
        datetime.strptime(time_str, "%H:%M")
    except ValueError:
        raise ValueError(f"Invalid time: '{time_str}' (expected HH:MM)")
    return (
        f'schtasks /Create /SC DAILY /TN BudgetBackup '
        f'/TR "python {script_path}" /ST {time_str}'
    )


# --- TEST ---
print(build_cron_schedule(30, 18, "/home/user/backup.py"))
print(build_task_scheduler_command("18:30", "C:/scripts/backup.py"))

---
## Exercise 6: Automation Suite
**Objective:** Combine parsing, loading JSON, listing expenses, backups, and summaries into one mini CLI application.

Build a tool with these subcommands:
- `list` → print all expenses
- `summary` → show totals by category
- `backup` → make a timestamped backup of the JSON file

**Tasks:**
1. `load_json_expenses(filename)` → Safely load a JSON list of expense dictionaries.
2. `build_suite_parser()` → Create an `argparse` parser with subcommands:
   - `list`
   - `summary`
   - `backup`
   Also add a common `filename` argument.
3. `run_suite(argv=None)` → Parse the arguments and perform the correct action.
4. Reuse functions from earlier exercises when possible.

**Run the setup cell below first.**


In [ ]:
import json

suite_data = [
    {"date": "2026-10-01", "category": "Food", "amount": 18.50, "note": "Lunch"},
    {"date": "2026-10-02", "category": "Transport", "amount": 2.80, "note": "Metro"},
    {"date": "2026-10-03", "category": "Food", "amount": 21.30, "note": "Dinner"},
    {"date": "2026-10-04", "category": "Books", "amount": 45.00, "note": "Python book"},
]

with open("expenses_suite.json", "w") as file:
    json.dump(suite_data, file, indent=2)

print("Created expenses_suite.json")

In [ ]:
import argparse
import json

def load_json_expenses(filename):
    """Safely load and validate a JSON expense list."""
    try:
        with open(filename) as file:
            data = json.load(file)
    except FileNotFoundError:
        print(f"Error: '{filename}' not found.")
        return []
    except json.JSONDecodeError:
        print(f"Error: '{filename}' is not valid JSON.")
        return []

    # Ensure the top-level object is a list
    if not isinstance(data, list):
        print(f"Error: '{filename}' must contain a list of expenses.")
        return []
    return data

def build_suite_parser():
    """Return an argparse parser with subcommands."""
    parser = argparse.ArgumentParser(description="Expense automation suite")
    parser.add_argument("filename", help="JSON file with expenses")

    subparsers = parser.add_subparsers(dest="command", required=True)
    subparsers.add_parser("list", help="Print all expenses")
    subparsers.add_parser("summary", help="Show totals by category")
    subparsers.add_parser("backup", help="Make a timestamped backup")
    return parser

def run_suite(argv=None):
    """Run the automation suite and return a message or report."""
    parser = build_suite_parser()
    args = parser.parse_args(argv)

    # backup does not need to load the data
    if args.command == "backup":
        destination = backup_file(args.filename)
        if destination is None:
            return "Backup failed."
        return f"Backup created: {destination}"

    expenses = load_json_expenses(args.filename)

    if args.command == "list":
        # Reuse format_expenses from Exercise 2
        return format_expenses(expenses)

    if args.command == "summary":
        # Reuse summarize_by_category from Exercise 4
        summary = summarize_by_category(expenses)
        if not summary:
            return "No expenses to summarize."
        lines = [f"{category:<12}{total:>10.2f}" for category, total in sorted(summary.items())]
        return "\n".join(lines)


# --- TEST ---
print(run_suite(["expenses_suite.json", "list"]))
print()
print(run_suite(["expenses_suite.json", "summary"]))
print()
print(run_suite(["expenses_suite.json", "backup"]))